# Experiments

### Colab Setup

In [1]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "anthropic",
            "gensim",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

    # the anthropic SDK reads the key from the environment
    tokens = dict(
        ln.strip().split("=", 1)
        for ln in open("/content/drive/MyDrive/thesis/tokens.env")
        if "=" in ln and not ln.startswith("#")
    )
    os.environ["ANTHROPIC_API_KEY"] = tokens["ANTHROPIC"]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Key Imports

In [2]:

import torch

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.loader_twd_labelled import load_splits
from models.benchmarks import always_neutral, random_guess
from models.bow import bow
from models.frozen_probe import probe
from models.llm_zero_shot import classify
from models.plm_finetune import finetune
from models.rule_based import rule_model
from models.word2vec import word2vec
from utils.results import already_done, run_seeds, save_result

OUT = RESULTS_DIR / "results.csv"
# flip to "chrono" to rerun everything on the chronological split
SPLIT = "benchmark"
CORPUS = "twd" if SPLIT == "benchmark" else "twd-chrono"
# chrono's partition is fixed, so the seed only varies weight init -- one run is enough.
# 78516 rather than 5768: 5768 collapsed roberta-large on the smaller chrono train set.
SEEDS = SHAH_SEEDS if SPLIT == "benchmark" else (78516,)
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


def run(model, predict):
    """run_seeds with this notebook's OUT / SEEDS / CORPUS / SPLIT / FORCE."""
    run_seeds(OUT, model, predict, SEEDS, CORPUS, SPLIT, FORCE)


results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


## Chance benchmarks

In [3]:
run("random", lambda train, test, seed: random_guess(test, seed))
run("always-neutral", lambda train, test, seed: always_neutral(test))

random seed 5768: already done, skipping
random seed 78516: already done, skipping
random seed 944601: already done, skipping
always-neutral seed 5768: already done, skipping
always-neutral seed 78516: already done, skipping
always-neutral seed 944601: already done, skipping


## Rule-based

In [4]:
run("rule-based", lambda train, test, seed: rule_model(test["sentence"].to_list()))

rule-based seed 5768: already done, skipping
rule-based seed 78516: already done, skipping
rule-based seed 944601: already done, skipping


## Bag-of-words

In [5]:
# raw text, unigrams -- six preprocessing variants moved macro-F1 by <0.008
run("bow", lambda train, test, seed: bow(train, test, seed=seed))

bow seed 5768: already done, skipping
bow seed 78516: already done, skipping
bow seed 944601: already done, skipping


## Word2Vec

In [6]:
# first call downloads 1.6GB of GoogleNews vectors, then cached
run("word2vec", lambda train, test, seed: word2vec(train, test, seed=seed))

word2vec seed 5768: already done, skipping
word2vec seed 78516: already done, skipping
word2vec seed 944601: already done, skipping


## Frozen encoders

In [7]:
for name, cfg in SHAH_PLM.items():
    run(
        f"frozen:{name}",
        lambda train, test, seed, cfg=cfg: probe(
            train,
            test,
            model_name=cfg["model_name"],
            max_len=cfg.get("max_len", 256),
            device=DEVICE,
            seed=seed,
        ),
    )

frozen:bert-base-uncased seed 5768: already done, skipping
frozen:bert-base-uncased seed 78516: already done, skipping
frozen:bert-base-uncased seed 944601: already done, skipping
frozen:bert-large-uncased seed 5768: already done, skipping
frozen:bert-large-uncased seed 78516: already done, skipping
frozen:bert-large-uncased seed 944601: already done, skipping
frozen:roberta-base seed 5768: already done, skipping
frozen:roberta-base seed 78516: already done, skipping
frozen:roberta-base seed 944601: already done, skipping
frozen:roberta-large seed 5768: already done, skipping
frozen:roberta-large seed 78516: already done, skipping
frozen:roberta-large seed 944601: already done, skipping
frozen:flang-bert seed 5768: already done, skipping
frozen:flang-bert seed 78516: already done, skipping
frozen:flang-bert seed 944601: already done, skipping
frozen:flang-roberta seed 5768: already done, skipping
frozen:flang-roberta seed 78516: already done, skipping
frozen:flang-roberta seed 944601: 

## Fine-tuned encoders

In [8]:
for MODEL in SHAH_PLM:
    cfg = SHAH_PLM[MODEL]

    for seed in SEEDS:
        if already_done(OUT, force=FORCE, model=MODEL, corpus=CORPUS, seed=seed):
            print(f"{MODEL} seed {seed}: already done, skipping")
            continue

        train, test = load_splits(SPLIT, seed=seed)
        model, tok_, metrics = finetune(
            train,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            max_len=cfg.get("max_len", 256),
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )

        save_result(
            OUT,
            model=MODEL,
            corpus=CORPUS,
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(
            f"{MODEL} seed {seed}: weighted={metrics['test_f1']:.4f}  macro={metrics['test_macro_f1']:.4f}"
        )

        # drop the model before the next seed builds one -- two large models plus
        # AdamW state don't fit on a 22GB L4
        del model, tok_
        torch.cuda.empty_cache()

bert-base-uncased seed 5768: already done, skipping
bert-base-uncased seed 78516: already done, skipping
bert-base-uncased seed 944601: already done, skipping
bert-large-uncased seed 5768: already done, skipping
bert-large-uncased seed 78516: already done, skipping
bert-large-uncased seed 944601: already done, skipping
roberta-base seed 5768: already done, skipping
roberta-base seed 78516: already done, skipping
roberta-base seed 944601: already done, skipping
roberta-large seed 5768: already done, skipping
roberta-large seed 78516: already done, skipping
roberta-large seed 944601: already done, skipping
flang-bert seed 5768: already done, skipping
flang-bert seed 78516: already done, skipping
flang-bert seed 944601: already done, skipping
flang-roberta seed 5768: already done, skipping
flang-roberta seed 78516: already done, skipping
flang-roberta seed 944601: already done, skipping


## Zero-shot LLMs

In [9]:
# no training, ~500 API calls per model per seed
for LLM in ["claude-haiku-4-5", "claude-sonnet-5", "claude-opus-4-8", "claude-fable-5"]:
    run(
        f"zero-shot:{LLM}",
        lambda train, test, seed, LLM=LLM: classify(
            test["sentence"].to_list(), model=LLM
        ),
    )

zero-shot:claude-haiku-4-5 seed 5768: already done, skipping
zero-shot:claude-haiku-4-5 seed 78516: already done, skipping
zero-shot:claude-haiku-4-5 seed 944601: already done, skipping
zero-shot:claude-sonnet-5 seed 5768: already done, skipping
zero-shot:claude-sonnet-5 seed 78516: already done, skipping
zero-shot:claude-sonnet-5 seed 944601: already done, skipping
zero-shot:claude-opus-4-8 seed 5768: already done, skipping
zero-shot:claude-opus-4-8 seed 78516: already done, skipping
zero-shot:claude-opus-4-8 seed 944601: already done, skipping
zero-shot:claude-fable-5 seed 5768: already done, skipping
zero-shot:claude-fable-5 seed 78516: already done, skipping
zero-shot:claude-fable-5 seed 944601: already done, skipping
